Given historical test execution data, predict failure probability and prioritize test cases.

In [12]:
import pandas as pd

data = {
    "test_case": ["TC1","TC2","TC3","TC4","TC5","TC6","TC7","TC8","TC9","TC10","TC11","TC12"],
    "module": ["Login","Login","Search","Search","Payment","Payment","Login","Search","Payment","Login","Search","Payment"],
    "execution_time": [2,3,5,4,7,6,2.5,4.5,6.5,3,5.5,7.5],
    "priority": ["High","Low","Medium","High","High","Medium","Low","Medium","High","Low","Medium","High"],
    "environment": ["QA","QA","UAT","QA","Prod","UAT","QA","Prod","Prod","QA","UAT","Prod"],
    "status": ["Pass","Fail","Pass","Fail","Fail","Pass","Pass","Fail","Fail","Pass","Pass","Fail"]
}

df = pd.DataFrame(data)

In [14]:
df

,test_case,module,execution_time,priority,environment,status
0,TC1,Login,2.0,High,QA,Pass
1,TC2,Login,3.0,Low,QA,Fail
2,TC3,Search,5.0,Medium,UAT,Pass
3,TC4,Search,4.0,High,QA,Fail
4,TC5,Payment,7.0,High,Prod,Fail
5,TC6,Payment,6.0,Medium,UAT,Pass
6,TC7,Login,2.5,Low,QA,Pass
7,TC8,Search,4.5,Medium,Prod,Fail
8,TC9,Payment,6.5,High,Prod,Fail
9,TC10,Login,3.0,Low,QA,Pass


In [18]:
df["status"] = df["status"].str.strip()

In [20]:
df["status"] = df["status"].map({"Pass": 0, "Fail": 1})

In [22]:
df

,test_case,module,execution_time,priority,environment,status
0,TC1,Login,2.0,High,QA,0
1,TC2,Login,3.0,Low,QA,1
2,TC3,Search,5.0,Medium,UAT,0
3,TC4,Search,4.0,High,QA,1
4,TC5,Payment,7.0,High,Prod,1
5,TC6,Payment,6.0,Medium,UAT,0
6,TC7,Login,2.5,Low,QA,0
7,TC8,Search,4.5,Medium,Prod,1
8,TC9,Payment,6.5,High,Prod,1
9,TC10,Login,3.0,Low,QA,0


In [24]:
df = pd.get_dummies(df, columns=["module","priority","environment"], drop_first=True)

In [26]:
df

,test_case,execution_time,status,module_Payment,module_Search,priority_Low,priority_Medium,environment_QA,environment_UAT
0,TC1,2.0,0,False,False,False,False,True,False
1,TC2,3.0,1,False,False,True,False,True,False
2,TC3,5.0,0,False,True,False,True,False,True
3,TC4,4.0,1,False,True,False,False,True,False
4,TC5,7.0,1,True,False,False,False,False,False
5,TC6,6.0,0,True,False,False,True,False,True
6,TC7,2.5,0,False,False,True,False,True,False
7,TC8,4.5,1,False,True,False,True,False,False
8,TC9,6.5,1,True,False,False,False,False,False
9,TC10,3.0,0,False,False,True,False,True,False


In [28]:
X = df.drop(["status","test_case"], axis=1)
y = df["status"]

In [50]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [52]:
#logistic regression
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

LogisticRegression()

In [54]:
#decisiontree
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=4)
dt.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=4)

In [56]:
#RandomForest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [58]:
from sklearn.metrics import classification_report

print("Logistic Regression\n", classification_report(y_test, lr.predict(X_test),zero_division=0))
print("Decision Tree\n", classification_report(y_test, dt.predict(X_test),zero_division=0))
print("Random Forest\n", classification_report(y_test, rf.predict(X_test),zero_division=0))

Logistic Regression
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.33      1.00      0.50         1

    accuracy                           0.33         3
   macro avg       0.17      0.50      0.25         3
weighted avg       0.11      0.33      0.17         3

Decision Tree
               precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       0.50      1.00      0.67         1

    accuracy                           0.67         3
   macro avg       0.75      0.75      0.67         3
weighted avg       0.83      0.67      0.67         3

Random Forest
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.33      1.00      0.50         1

    accuracy                           0.33         3
   macro avg       0.17      0.50      0.25         3
weighted avg       0.11

In [60]:
df["fail_prob"] = dt.predict_proba(X)[:,1]

In [64]:
df

,test_case,execution_time,status,module_Payment,module_Search,priority_Low,priority_Medium,environment_QA,environment_UAT,fail_prob
0,TC1,2.0,0,False,False,False,False,True,False,0.0
1,TC2,3.0,1,False,False,True,False,True,False,1.0
2,TC3,5.0,0,False,True,False,True,False,True,0.0
3,TC4,4.0,1,False,True,False,False,True,False,1.0
4,TC5,7.0,1,True,False,False,False,False,False,1.0
5,TC6,6.0,0,True,False,False,True,False,True,0.0
6,TC7,2.5,0,False,False,True,False,True,False,0.0
7,TC8,4.5,1,False,True,False,True,False,False,1.0
8,TC9,6.5,1,True,False,False,False,False,False,1.0
9,TC10,3.0,0,False,False,True,False,True,False,1.0


In [66]:
priority_df = df.sort_values(by="fail_prob", ascending=False)

print(priority_df[["test_case","fail_prob"]])

   test_case  fail_prob
1        TC2        1.0
3        TC4        1.0
4        TC5        1.0
7        TC8        1.0
8        TC9        1.0
9       TC10        1.0
11      TC12        1.0
0        TC1        0.0
2        TC3        0.0
5        TC6        0.0
6        TC7        0.0
10      TC11        0.0


In [68]:
def priority_label(p):
    if p > 0.8:
        return "High"
    elif p > 0.5:
        return "Medium"
    else:
        return "Low"

priority_df["priority"] = priority_df["fail_prob"].apply(priority_label)

In [48]:
priority_df

,test_case,execution_time,status,module_Payment,module_Search,priority_Low,priority_Medium,environment_QA,environment_UAT,fail_prob,priority
1,TC2,3.0,1,False,False,True,False,True,False,1.0,High
3,TC4,4.0,1,False,True,False,False,True,False,1.0,High
4,TC5,7.0,1,True,False,False,False,False,False,1.0,High
7,TC8,4.5,1,False,True,False,True,False,False,1.0,High
8,TC9,6.5,1,True,False,False,False,False,False,1.0,High
9,TC10,3.0,0,False,False,True,False,True,False,1.0,High
11,TC12,7.5,1,True,False,False,False,False,False,1.0,High
0,TC1,2.0,0,False,False,False,False,True,False,0.0,Low
2,TC3,5.0,0,False,True,False,True,False,True,0.0,Low
5,TC6,6.0,0,True,False,False,True,False,True,0.0,Low
